# 08 · Sensitivity analysis: duplicate rows
The main pipeline drops exact duplicate rows before splitting (9.5 % of the data). The duplicates are not copy errors, they are different survey respondents with identical answers; the 21 features are almost all binary/ordinal, 67 duplicate groups even have conflicting labels.
Here we check how much the decision matters by running the alternative protocol: split first, keep duplicates, and measuring how many test people have an exact twin in train.

In [ ]:
import sys; sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import warnings
from functools import partialmethod
from tqdm import tqdm; tqdm.__init__ = partialmethod(tqdm.__init__, disable=True)
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
from src.preprocess import get_splits
from src.evaluation import threshold_for_recall
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid")

In [2]:
def fit_models(X_tr, y_tr, X_va, y_va):
    lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X_tr, y_tr)
    gb = lgb.LGBMClassifier(n_estimators=3000, learning_rate=0.03, num_leaves=15, min_child_samples=100, subsample=0.8, subsample_freq=1,
                            colsample_bytree=0.8, reg_lambda=5.0, random_state=42, verbose=-1)
    gb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric="binary_logloss", callbacks=[lgb.early_stopping(100, verbose=False)])
    return lr, gb

res, models, splits = [], {}, {}
for proto, drop in [("A · drop duplicates (main)", True), ("B · keep duplicates", False)]:
    X_tr, X_va, X_te, y_tr, y_va, y_te = get_splits(drop_duplicates=drop); splits[proto] = (X_tr, X_va, X_te, y_tr, y_va, y_te)
    lr, gb = fit_models(X_tr, y_tr, X_va, y_va); models[proto] = (lr, gb)
    for name, m in [("LogReg", lr), ("LightGBM", gb)]:
        pv, pt = m.predict_proba(X_va)[:, 1], m.predict_proba(X_te)[:, 1]; t = threshold_for_recall(y_va, pv, 0.80)
        res.append({"protocol": proto, "model": name, "n_total": len(X_tr) + len(X_va) + len(X_te), "prevalence": y_te.mean(), "ROC-AUC": roc_auc_score(y_te, pt),
                    "PR-AUC": average_precision_score(y_te, pt), "precision@recall80": precision_score(y_te, pt >= t), "recall": recall_score(y_te, pt >= t)})
pd.DataFrame(res).round(3)

,protocol,model,n_total,prevalence,ROC-AUC,PR-AUC,precision@recall80,recall
0,A · drop duplicates (main),LogReg,229474,0.153,0.807,0.404,0.304,0.797
1,A · drop duplicates (main),LightGBM,229474,0.153,0.818,0.442,0.313,0.799
2,B · keep duplicates,LogReg,253680,0.139,0.821,0.398,0.299,0.802
3,B · keep duplicates,LightGBM,253680,0.139,0.830,0.431,0.304,0.808


## Leakage: how many test people have an exact twin in train?
twin = an identical feature vector appears in the training set. If twins are easier to predict, the metrics under protocol B are optimistic.

In [3]:
X_tr, X_va, X_te, y_tr, y_va, y_te = splits["B · keep duplicates"]; gb = models["B · keep duplicates"][1]
h = lambda X: pd.util.hash_pandas_object(X, index=False)
twin = h(X_te).isin(set(h(X_tr))).values
pt = gb.predict_proba(X_te)[:, 1]
print(f"test people with an exact twin in train (protocol B): {twin.mean():.1%}")
tab = pd.DataFrame({"n": [twin.sum(), (~twin).sum()], "prevalence": [y_te.values[twin].mean(), y_te.values[~twin].mean()],
                    "ROC-AUC": [roc_auc_score(y_te.values[twin], pt[twin]), roc_auc_score(y_te.values[~twin], pt[~twin])]}, index=["has twin in train", "no twin"]).round(3)
tab

test people with an exact twin in train (protocol B): 12.6%


,n,prevalence,ROC-AUC
has twin in train,4785,0.051,0.839
no twin,33267,0.152,0.822


## Do the explanations change?
Global mean |SHAP| (2,000 test rows) of the LightGBM under both protocols, 21 features.

In [4]:
imp = {}
for proto, drop in [("A · drop duplicates (main)", True), ("B · keep duplicates", False)]:
    X_tr, X_va, X_te, *_ = splits[proto]; gb = models[proto][1]
    sv = shap.TreeExplainer(gb, data=X_tr.sample(100, random_state=0), feature_perturbation="interventional", model_output="raw")(X_te.sample(2000, random_state=0)).values
    imp[proto] = pd.Series(np.abs(sv).mean(0), index=X_tr.columns)
imp = pd.DataFrame(imp).sort_values("A · drop duplicates (main)", ascending=False)
print("Spearman of global SHAP rankings A vs B:", round(spearmanr(imp.iloc[:, 0], imp.iloc[:, 1])[0], 3))
imp.round(3).head(10)


 56%|===========         | 1118/2000 [00:11<00:08]       


 61%|============        | 1224/2000 [00:12<00:07]       


 66%|=============       | 1327/2000 [00:13<00:06]       


 71%|==============      | 1429/2000 [00:14<00:05]       


 77%|===============     | 1533/2000 [00:15<00:04]       


 82%|================    | 1637/2000 [00:16<00:03]       

... (7 outputs left out)


 89%|==================  | 1781/2000 [00:15<00:01]       


 95%|=================== | 1905/2000 [00:16<00:00]       

Spearman of global SHAP rankings A vs B: 0.942


,A · drop duplicates (main),B · keep duplicates
GenHlth,0.506,0.561
Age,0.387,0.415
BMI,0.368,0.406
HighBP,0.347,0.343
HighChol,0.273,0.284
Sex,0.121,0.134
CholCheck,0.112,0.051
Income,0.112,0.137
MentHlth,0.093,0.057
HeartDiseaseorAttack,0.085,0.056


## Conclusion

- **Keeping duplicates makes the model look better than it actually is:** LightGBM gets a ROC-AUC of 0.818 when duplicates are removed, compared to 0.830 when they are kept. For Logistic Regression, the scores are 0.807 vs 0.821. However, the prevalence also differs between the two approaches (15.3% vs 13.9%), so part of this difference may be due to the different composition of the datasets.

- **There is clear evidence of data leakage with protocol B:** 12.6% of the people in the test set (4,785 individuals) have an exact duplicate in the training set. These people are easier for the model to predict, with a ROC-AUC of 0.839 compared to 0.822 for people without a duplicate.

- **The explanations are quite similar, but not exactly the same:** the Spearman correlation between the global SHAP rankings is 0.94. The top 5 most important features remain the same, although some change their importance. For example, CholCheck goes from 0.11 to 0.05, and MentHlth from 0.09 to 0.06.

- **For these reasons, we decided to keep protocol A (removing duplicates) as our main approach:** it is the more conservative option, and the main conclusions about which features are important remain the same regardless of which protocol is used.